
---
## Section 1 - Problem Identification & Mathematical Formulation

### 1.1 Problem Statement

The **Valorant Champions Tour (VCT)** is Riot Games' premier global esports circuit, featuring professional teams competing across regional leagues (Americas, EMEA, Pacific, China) and international events. Each season consists of Kickoff tournaments, Stage 1 & 2 leagues, Masters events, and the World Championship. Hundreds of millions of dollars in prize money, sponsorships, and player salaries depend on match outcomes.

**The Problem:** Given two competing VCT teams and their historical performance records at the time of the match, can we predict which team will win?

This is a **Binary Classification** task: the model must assign each match to one of two classes:
- **Class 1:** Team A wins (`team_a_won = 1`)
- **Class 0:** Team A loses / Team B wins (`team_a_won = 0`)

### 1.2 Stakeholders & Their Needs

| Stakeholder | Need |
|-------------|------|
| **Coaching Staff** | Prioritise preparation against teams the model flags as high-threat; understand which historical metrics (win rate, rating differential) drive predictions |
| **Team Managers / GMs** | Roster construction decisions - identify under-valued players using the `hist_avg_rating` feature to find teams trending upward |
| **Tournament Organisers** | Seeding and bracket design based on expected match competitiveness |
| **Analysts & Statisticians** | Feature importance rankings to identify which competitive KPIs matter most |
| **Commentators & Media** | Pre-match win probability percentages for broadcast narratives |
| **Betting & Fantasy Platforms** | Calibrated probability estimates (ROC-AUC maximisation) for line-setting |

### 1.3 Mathematical Formulation

#### 1.3.1 Feature Space

Each training example is a match observation. We define the **feature vector** $\mathbf{x} \in \mathbb{R}^{13}$ as:

$$
\mathbf{x} = \bigl[
  x_1,\; x_2,\; x_3,\; x_4,\; x_5,\; x_6,\; x_7,\; x_8,\; x_9,\; x_{10},\; x_{11},\; x_{12},\; x_{13}
\bigr]^{\top}
$$

where each component is defined as:

| Index | Feature | Definition |
|-------|---------|------------|
| $x_1$ | `ta_hist_win_rate` | Team A's historical win rate over all prior matches: $\frac{\sum_{k<t} \mathbf{1}[\text{A wins match } k]}{\lvert \mathcal{H}_A \rvert}$ |
| $x_2$ | `tb_hist_win_rate` | Team B's historical win rate over all prior matches |
| $x_3$ | `hist_win_rate_diff` | $x_1 - x_2$ |
| $x_4$ | `ta_hist_avg_rating` | Team A's historical mean player rating: $\frac{1}{\lvert \mathcal{H}_A \rvert}\sum_{k<t} r_{A,k}$ |
| $x_5$ | `tb_hist_avg_rating` | Team B's historical mean player rating |
| $x_6$ | `hist_rating_diff` | $x_4 - x_5$ |
| $x_7$ | `ta_hist_map_win_pct` | Team A's cumulative map win percentage: $\frac{\text{maps won}_A}{\text{maps played}_A}$ |
| $x_8$ | `tb_hist_map_win_pct` | Team B's cumulative map win percentage |
| $x_9$ | `map_win_pct_diff` | $x_7 - x_8$ |
| $x_{10}$ | `stage_stakes` | Ordinal match importance: 1 = league play, 2 = group/knockout, 3 = playoffs/finals |
| $x_{11}$ | `is_elimination_match` | $\mathbf{1}[\text{match type} \in \{\text{elimination, lower bracket, decider}\}]$ |
| $x_{12}$ | `is_grand_final` | $\mathbf{1}[\text{match type} = \text{Grand Final}]$ |
| $x_{13}$ | `ta_ban_first` | $\mathbf{1}[\text{Team A holds ban slot \#1 in draft}]$ |

#### 1.3.2 Label Space

The binary target variable is:

$$
y \in \mathcal{Y} = \{0, 1\}, \quad y = \mathbf{1}[\text{Team A wins the match}]
$$

#### 1.3.3 Hypothesis Class

We seek a classifier $f_{\boldsymbol{\theta}} : \mathbb{R}^{13} \to [0,1]$ that maps a feature vector to the probability that Team A wins:

$$
\hat{p} = f_{\boldsymbol{\theta}}(\mathbf{x}) = P(y = 1 \mid \mathbf{x};\, \boldsymbol{\theta})
$$

The predicted label is obtained via a decision threshold $\tau = 0.5$:

$$
\hat{y} = \mathbf{1}[\hat{p} \geq \tau]
$$

#### 1.3.4 Objective Function - Binary Cross-Entropy (Log-Loss)

The **primary training objective** $Z$ is the minimisation of the **Binary Cross-Entropy loss** over the $n$ training examples:

$$
\boxed{
Z(\boldsymbol{\theta}) = \underset{\boldsymbol{\theta}}{\min} \; \mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{n} \sum_{i=1}^{n} \Bigl[ y^{(i)} \log \hat{p}^{(i)} + \bigl(1 - y^{(i)}\bigr) \log\bigl(1 - \hat{p}^{(i)}\bigr) \Bigr]
}
$$

where:
- $n$ is the number of training matches
- $y^{(i)} \in \{0, 1\}$ is the true label for match $i$
- $\hat{p}^{(i)} = f_{\boldsymbol{\theta}}(\mathbf{x}^{(i)}) \in (0, 1)$ is the predicted win probability
- $\boldsymbol{\theta}$ represents all learnable parameters of the model

**Intuition:** The log-loss penalises confident wrong predictions exponentially. A model that outputs $\hat{p} = 0.99$ but $y = 0$ incurs a loss of $\approx 4.6$, versus $\approx 0.01$ for a correct confident prediction. This forces the model to be well-calibrated - a critical property for esports stakeholders who rely on win probabilities, not just binary predictions.

#### 1.3.5 Evaluation Metrics

While $Z$ is the training objective, we **evaluate** models on held-out data using:

$$
\text{Accuracy} = \frac{TP + TN}{n_{\text{test}}}, \quad
F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}, \quad
\text{ROC-AUC} = \int_0^1 \text{TPR}(t)\, d\,\text{FPR}(t)
$$

**F1 Score** is the primary evaluation metric (harmonic mean of Precision and Recall), as it handles the mild class imbalance (52.5% Team A wins, 47.5% Team B wins) more robustly than raw accuracy. **ROC-AUC** measures probability calibration quality - essential for bookmakers and analysts.
